In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

In [ ]:
start_day = 1223
end_day = 1307

def get_week_idx(day_num, start_day=1223):
    return (day_num - start_day) // 7

marketplace_files = []
marketplace_events_list = []

for day in range(start_day, end_day + 1):
    day_str = f"{day:05d}"
    file_pattern = f'dataset/small/marketplace/events/{day_str}.pq'
    files = glob.glob(file_pattern)
    
    for file in files:
        df = pd.read_parquet(file)
        df['idx_time'] = get_week_idx(day)  # Add week index
        marketplace_events_list.append(df)

marketplace_events = pd.concat(marketplace_events_list, ignore_index=True)

train_data = marketplace_events[
    marketplace_events['idx_time'].isin([0, 1, 2, 3, 4, 5, 6])
]

val_data = marketplace_events[
    marketplace_events['idx_time'].isin([8])
]

test_data = marketplace_events[
    marketplace_events['idx_time'].isin([10, 11, 12])
]

print("Marketplace weeks distribution:")
print(marketplace_events['idx_time'].value_counts().sort_index())

Marketplace weeks distribution:
idx_time
0     4534537
1     4616696
2     4497943
3     4886528
4     4703509
5     4581339
6     4835004
7     4852298
8     5158296
9     4663645
10    3347278
11    3548098
12     539733
Name: count, dtype: int64


In [3]:
marketplace_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,idx_time
0,1223 days 00:00:00.147515,70777849,nfmcg_7326721,u2i,view,android,0
1,1223 days 00:00:00.201888,55534656,nfmcg_27530868,u2i,view,ios,0
2,1223 days 00:00:00.395625,75229774,nfmcg_11875472,other,view,android,0
3,1223 days 00:00:00.666633,44037776,nfmcg_19335823,u2i,view,android,0
4,1223 days 00:00:00.713864,35039031,nfmcg_27582454,search,view,ios,0


In [4]:
marketplace_events['subdomain'].value_counts()

subdomain
u2i        29726435
catalog     6936825
other       6664069
i2i         5831291
search      5589683
Name: count, dtype: int64

In [5]:
marketplace_events['action_type'].value_counts()

action_type
view        53095195
click        1446502
clickout      206606
like           16601
Name: count, dtype: int64

In [ ]:
train_data['idx_item'] = train_data['item_id'].str.split('_').str[1]
train_data['item_id'] = train_data['item_id'].str.split('_').str[0]
train_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
train_data['treated'] = train_data['treated'].isin(['u2i', 'i2i']).astype(int)
train_data['outcome'] = train_data['outcome'].isin(['click', 'clickout', 'like']).astype(int)
train_data.drop(['os'], axis = 1, inplace = True)
train_data['personal_popular'] = train_data.groupby(['idx_user', 'idx_item'])['outcome'].transform('sum')
train_data['personal_popular'].fillna(0, inplace=True)
train_data = train_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
train_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,70777849,7326721,nfmcg,1,0,0,1223 days 00:00:00.147515,0
1,55534656,27530868,nfmcg,1,0,0,1223 days 00:00:00.201888,0
2,75229774,11875472,nfmcg,0,0,0,1223 days 00:00:00.395625,0
3,44037776,19335823,nfmcg,1,0,0,1223 days 00:00:00.666633,0
4,35039031,27582454,nfmcg,0,0,0,1223 days 00:00:00.713864,0


In [7]:
df_merge = train_data[['idx_user', 'idx_item', 'personal_popular']]

In [8]:
test_data['idx_item'] = test_data['item_id'].str.split('_').str[1]
test_data['item_id'] = test_data['item_id'].str.split('_').str[0]
test_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
test_data['treated'] = test_data['treated'].isin(['u2i', 'i2i']).astype(int)
test_data['outcome'] = test_data['outcome'].isin(['click', 'clickout', 'like']).astype(int)
test_data.drop(['os'], axis = 1, inplace = True)
test_data = pd.merge(test_data, df_merge, on=['idx_user', 'idx_item'], how='left')
test_data = test_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
test_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,52232773,15699448,nfmcg,1,0,NaN,1293 days 00:00:00.062969,10
1,31069502,1250676,nfmcg,1,0,NaN,1293 days 00:00:00.314868,10
2,44830368,15699448,nfmcg,1,0,NaN,1293 days 00:00:00.315447,10
3,62935244,19354294,nfmcg,0,0,NaN,1293 days 00:00:00.606837,10
4,82057795,12755858,nfmcg,0,0,NaN,1293 days 00:00:01.039481,10


In [9]:
val_data['idx_item'] = val_data['item_id'].str.split('_').str[1]
val_data['item_id'] = val_data['item_id'].str.split('_').str[0]
val_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
val_data['treated'] = val_data['treated'].isin(['u2i', 'i2i']).astype(int)
val_data['outcome'] = val_data['outcome'].isin(['click', 'clickout', 'like']).astype(int)
val_data.drop(['os'], axis = 1, inplace = True)
val_data = pd.merge(val_data, df_merge, on=['idx_user', 'idx_item'], how='left')
val_data = val_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
val_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,1395414,4854616,nfmcg,1,0,NaN,1279 days 00:00:00.037405,8
1,35307915,7696108,nfmcg,1,0,NaN,1279 days 00:00:00.037405,8
2,56330098,3273133,nfmcg,1,0,NaN,1279 days 00:00:00.219769,8
3,12051397,25928977,nfmcg,1,0,NaN,1279 days 00:00:00.655919,8
4,38103237,17648976,nfmcg,0,0,NaN,1279 days 00:00:01.306735,8


In [10]:
df_with_features_train = train_data.copy()

df_with_features_train['hour'] = df_with_features_train['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_train.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_train.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_train = df_with_features_train.merge(item_popularity, on='idx_item', how='left')
df_with_features_train = df_with_features_train.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_train.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_train = df_with_features_train.merge(user_activity, on='idx_user', how='left')

feature_cols = [
    'hour', 
    'item_popularity', 
    'item_cvr',
    'user_unique_items',
    'user_total_interactions',
    'user_conversion_rate'
]

df_with_features_train[feature_cols] = df_with_features_train[feature_cols].fillna(0)
prop_model = LogisticRegression(max_iter=1000)
X_prop = df_with_features_train[feature_cols]

prop_model.fit(X_prop, df_with_features_train['treated'])
df_with_features_train['propensity'] = prop_model.predict_proba(X_prop)[:, 1]
df_with_features_train['ipw_weight'] = np.where(
    df_with_features_train['treated'] == 1,
    1 / df_with_features_train['propensity'],
    1 / (1 - df_with_features_train['propensity'])
)
train_data['propensity'] = df_with_features_train['propensity'].copy()

treated_df = df_with_features_train[df_with_features_train['treated'] == 1]
control_df = df_with_features_train[df_with_features_train['treated'] == 0]

print(f"Treated samples: {len(treated_df)}")
print(f"Control samples: {len(control_df)}")
print(f"Positive outcomes: {df_with_features_train['outcome'].sum()}")

Treated samples: 21620482
Control samples: 11035074
Positive outcomes: 994212


In [11]:
df_with_features_test = test_data.copy()

df_with_features_test['hour'] = df_with_features_test['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_test.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_test.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_test = df_with_features_test.merge(item_popularity, on='idx_item', how='left')
df_with_features_test = df_with_features_test.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_test.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_test = df_with_features_test.merge(user_activity, on='idx_user', how='left')

df_with_features_test[feature_cols] = df_with_features_test[feature_cols].fillna(0)
X_prop = df_with_features_test[feature_cols]

df_with_features_test['propensity'] = prop_model.predict_proba(X_prop)[:, 1]
df_with_features_test['ipw_weight'] = np.where(
    df_with_features_test['treated'] == 1,
    1 / df_with_features_test['propensity'],
    1 / (1 - df_with_features_test['propensity'])
)
test_data['propensity'] = df_with_features_test['propensity'].copy()

In [12]:
df_with_features_val = val_data.copy()

df_with_features_val['hour'] = df_with_features_val['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_val.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_val.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_val = df_with_features_val.merge(item_popularity, on='idx_item', how='left')
df_with_features_val = df_with_features_val.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_val.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_val = df_with_features_val.merge(user_activity, on='idx_user', how='left')

df_with_features_val[feature_cols] = df_with_features_val[feature_cols].fillna(0)
X_prop = df_with_features_val[feature_cols]

df_with_features_val['propensity'] = prop_model.predict_proba(X_prop)[:, 1]
df_with_features_val['ipw_weight'] = np.where(
    df_with_features_val['treated'] == 1,
    1 / df_with_features_val['propensity'],
    1 / (1 - df_with_features_val['propensity'])
)
val_data['propensity'] = df_with_features_val['propensity'].copy()

In [13]:
feature_cols_with_prop = feature_cols + ['propensity']

In [14]:
model_treated = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)

model_treated.fit(treated_df[feature_cols_with_prop], treated_df['outcome'], sample_weight=treated_df['ipw_weight'])
df_with_features_train['y1_pred'] = model_treated.predict(df_with_features_train[feature_cols_with_prop])

In [15]:
model_control = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)

model_control.fit(control_df[feature_cols_with_prop], control_df['outcome'], sample_weight=control_df['ipw_weight'])
df_with_features_train['y0_pred'] = model_control.predict(df_with_features_train[feature_cols_with_prop])

In [2]:
train_data = pd.read_csv("data_train.csv")
test_data = pd.read_csv("data_test.csv")
val_data = pd.read_csv("data_vali.csv")

In [3]:
train_data['propensity'] = np.clip(train_data['propensity'], 0.0001, 0.9999)
test_data['propensity'] = np.clip(test_data['propensity'], 0.0001, 0.9999)
val_data['propensity'] = np.clip(val_data['propensity'], 0.0001, 0.9999)

In [ ]:
df_1 = df_with_features_train.copy()
df_1 = df_1.reset_index(drop=True)
train_data = train_data.reset_index(drop=True)
train_data['causal_effect'] = (df_1['y1_pred'] - df_1['y0_pred'])
train_data['causal_effect'] = train_data['causal_effect'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

In [ ]:
df_with_features_test['y1_pred'] = model_treated.predict(df_with_features_test[feature_cols_with_prop])
df_with_features_test['y0_pred'] = model_control.predict(df_with_features_test[feature_cols_with_prop])
df_1 = df_with_features_test.copy()
df_1 = df_1.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)
test_data['causal_effect'] = (df_1['y1_pred'] - df_1['y0_pred'])
test_data['causal_effect'] = test_data['causal_effect'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

In [ ]:
df_with_features_val['y1_pred'] = model_treated.predict(df_with_features_val[feature_cols_with_prop])
df_with_features_val['y0_pred'] = model_control.predict(df_with_features_val[feature_cols_with_prop])
df_1 = df_with_features_val.copy()
df_1 = df_1.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
val_data['causal_effect'] = (df_1['y1_pred'] - df_1['y0_pred'])
val_data['causal_effect'] = val_data['causal_effect'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

In [21]:
train_data = train_data[
    train_data['idx_time'].isin([0, 1, 2])
]

train_data['idx_time'] = 0
val_data['idx_time'] = 0
test_data['idx_time'] = 0

print(f"Train: {len(train_data):,} ({len(train_data)/len(marketplace_events)*100:.1f}%)")
print(f"Val: {len(val_data):,} ({len(val_data)/len(marketplace_events)*100:.1f}%)")
print(f"Test: {len(test_data):,} ({len(test_data)/len(marketplace_events)*100:.1f}%)")

Train: 13,649,176 (24.9%)
Val: 5,578,143 (10.2%)
Test: 8,156,941 (14.9%)


In [4]:
train_data.to_csv('data_train.csv')
val_data.to_csv('data_vali.csv')
test_data.to_csv('data_test.csv')

## Looking at max and min causal_effect that can be obtained

In [3]:
test_data = pd.read_csv('data_test.csv')
test_data_descending = test_data.sort_values(by=['idx_user', 'causal_effect'], ascending=False)
test_data_ascending = test_data.sort_values(by=['idx_user', 'causal_effect'], ascending=True)

In [4]:
df_ranking_head_10 = test_data_descending.groupby('idx_user').head(10)
df_ranking_tail_10 = test_data_ascending.groupby('idx_user').head(10)

print(f"Best CP@10: {np.nanmean(df_ranking_head_10.loc[:, 'causal_effect'].values):.6f}")
print(f"Worst CP@10: {np.nanmean(df_ranking_tail_10.loc[:, 'causal_effect'].values):.6f}")

Best CP@10: 0.146084
Worst CP@10: -0.133768


In [5]:
df_ranking_head_100 = test_data_descending.groupby('idx_user').head(100)
df_ranking_tail_100 = test_data_ascending.groupby('idx_user').head(100)

print(f"Best CP@100: {np.nanmean(df_ranking_head_100.loc[:, 'causal_effect'].values):.6f}")
print(f"Worst CP@100: {np.nanmean(df_ranking_tail_100.loc[:, 'causal_effect'].values):.6f}")

Best CP@100: 0.126219
Worst CP@100: -0.065385


In [7]:
def dcg_at_k(rank_k, x):
    k = min(rank_k, len(x)) 
    return np.sum(x[:k] / np.log2(np.arange(k) + 2))

In [8]:
print(f"Best CDCG: {float(np.nanmean(test_data_descending.groupby('idx_user')['causal_effect'].apply(lambda x: dcg_at_k(100000, x)))):.6f}")
print(f"Worst CDCG: {float(np.nanmean(test_data_ascending.groupby('idx_user')['causal_effect'].apply(lambda x: dcg_at_k(100000, x)))):.6f}")

Best CDCG: 0.702608
Worst CDCG: -0.097310
